# Airbnb Listings - Adapted Analysis Notebook

## Análisis del dataset Listings – Airbnb Barcelona 2025

Este notebook tiene como objetivo analizar el dataset de anuncios (`listings`) de Airbnb en Barcelona, con el fin de comprender la estructura de la oferta, detectar patrones relevantes y preparar los datos para su posterior almacenamiento en una base de datos SQL.

El análisis se centra especialmente en:
- Características de los alojamientos
- Perfil de los hosts
- Distribución territorial
- Situación regulatoria (licencias)
- Diferencias entre anuncios con y sin precio informado


In [1]:
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_rows", None)

### Carga del dataset

Se carga el archivo `listings.csv`, que contiene información a nivel de anuncio.  
La información surge de un scrapeo de datos obtenido en una fecha determinada del 2025 de listados brindados por Airbnb y cada fila representa un alojamiento publicado en dicha plataforma.

In [ ]:
listings = pd.read_csv("listings.csv")

### Exploración inicial del dataset

En este paso se revisa la estructura general del dataset, el número de registros, los tipos de datos y la presencia de valores nulos.  
Este análisis permite identificar posibles problemas de calidad de datos y definir los pasos de limpieza necesarios.

In [3]:
listings.head()

,id,listing_url,scrape_id,last_scraped,source,name,description,neighborhood_overview,picture_url,host_id,...,review_scores_communication,review_scores_location,review_scores_value,license,instant_bookable,calculated_host_listings_count,calculated_host_listings_count_entire_homes,calculated_host_listings_count_private_rooms,calculated_host_listings_count_shared_rooms,reviews_per_month
0,18674,https://www.airbnb.com/rooms/18674,20250914152803,2025-09-15,city scrape,Huge flat for 8 people close to Sagrada Familia,110m2 apartment to rent in Barcelona. Located ...,Apartment in Barcelona located in the heart of...,https://a0.muscache.com/pictures/13031453/413c...,71615,...,4.62,4.82,4.32,ESFCTU000008058000039706000000000000000HUTB-00...,t,26,26,0,0,0.34
1,23197,https://www.airbnb.com/rooms/23197,20250914152803,2025-09-14,city scrape,"Forum CCIB DeLuxe, Spacious, Large Balcony, relax",Beautiful and Spacious Apartment with Large Te...,"Strategically located in the Parc del Fòrum, a...",https://a0.muscache.com/pictures/miso/Hosting-...,90417,...,4.99,4.66,4.68,ESFCTU000008106000547162000000000000000000HUTB...,f,1,1,0,0,0.52
2,32711,https://www.airbnb.com/rooms/32711,20250914152803,2025-09-15,city scrape,Sagrada Familia area - Còrsega 1,A lovely two bedroom apartment only 250 m from...,What's nearby <br />This apartment is located...,https://a0.muscache.com/pictures/357b25e4-f414...,135703,...,4.89,4.89,4.47,HUTB-001722,f,2,2,0,0,0.88
3,34241,https://www.airbnb.com/rooms/34241,20250914152803,2025-09-15,city scrape,Stylish Top Floor Apartment - Ramblas Plaza Real,Located in close proximity to Plaza Real and L...,NaN,https://a0.muscache.com/pictures/2437facc-2fe7...,73163,...,4.68,4.73,4.23,Exempt,f,3,3,0,0,0.14
4,34981,https://www.airbnb.com/rooms/34981,20250914152803,2025-09-15,city scrape,VIDRE HOME PLAZA REAL on LAS RAMBLAS,Spacious apartment for large families or group...,"Located in Ciutat Vella in the Gothic Quarter,...",https://a0.muscache.com/pictures/c4d1723c-e479...,73163,...,4.72,4.65,4.46,ESFCTU000008119000093652000000000000000HUTB-00...,f,3,3,0,0,1.49


In [4]:
listings.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 19410 entries, 0 to 19409
Data columns (total 79 columns):
 #   Column                                        Non-Null Count  Dtype  
---  ------                                        --------------  -----  
 0   id                                            19410 non-null  int64  
 1   listing_url                                   19410 non-null  object 
 2   scrape_id                                     19410 non-null  int64  
 3   last_scraped                                  19410 non-null  object 
 4   source                                        19410 non-null  object 
 5   name                                          19410 non-null  object 
 6   description                                   18673 non-null  object 
 7   neighborhood_overview                         8986 non-null   object 
 8   picture_url                                   19410 non-null  object 
 9   host_id                                       19410 non-null 

### Distribución de disponibilidad anual

Se examinan los valores más frecuentes de `availability_365` con el fin de identificar patrones generales en la disponibilidad anual de los anuncios. Este análisis permite observar si existen concentraciones relevantes en valores específicos, como disponibilidad total (365 días) o ausencia completa de disponibilidad (0 días).

In [5]:
listings[[
    "availability_30",
    "availability_60",
    "availability_90",
    "availability_365"
]].describe()

,availability_30,availability_60,availability_90,availability_365
count,19410.000000,19410.000000,19410.000000,19410.000000
mean,8.424575,22.417414,40.611386,195.091190
std,9.881033,20.691874,31.801723,129.370782
min,0.000000,0.000000,0.000000,0.000000
25%,0.000000,0.000000,1.000000,72.000000
50%,5.000000,19.000000,43.000000,223.000000
75%,14.000000,39.000000,68.000000,318.000000
max,30.000000,60.000000,90.000000,365.000000


In [6]:
listings["availability_365"].value_counts().head(20)

availability_365
0      2734
365     771
364     345
269     197
318     158
257     150
319     147
349     140
341     135
335     125
270     110
348     110
340     109
317     106
226     102
343     100
333     100
332      99
363      99
288      98
Name: count, dtype: int64

In [7]:
df_av0 = listings[listings["availability_365"] == 0]
df_av0.shape

(2734, 79)

### Análisis de presencia de precio en anuncios sin disponibilidad

Se analiza la proporción de anuncios sin disponibilidad anual que no presentan precio informado. Este análisis permite identificar si la falta de disponibilidad está asociada a un comportamiento específico del mercado, como alojamientos no activos en el alquiler turístico tradicional.


In [8]:
df_av0["price"].isna().value_counts(normalize=True) * 100

price
True     95.57425
False     4.42575
Name: proportion, dtype: float64

### Eliminación de anuncios sin disponibilidad anual

Tras el análisis exploratorio de las variables de disponibilidad, se observó que los anuncios con `availability_365 = 0` presentan una proporción extremadamente alta de ausencia de precio informado. Este patrón sugiere que la mayoría de estos registros no forma parte del mercado activo de alquiler turístico durante el período analizado.

Dado que el objetivo principal del análisis se centra en el comportamiento del mercado activo, se decidió excluir estos anuncios del dataset de trabajo. Esta decisión busca evitar sesgos en los análisis posteriores relacionados con precios, ocupación y comportamiento del mercado, manteniendo únicamente aquellos anuncios con disponibilidad anual positiva.


In [9]:
listings = listings[listings["availability_365"] > 0].copy()

In [10]:
listings["availability_365"].min()

1

In [11]:
listings.shape # Se redujo el dataset de 19.410 registros a 16.676 registros

(16676, 79)

### Selección de variables de análisis

A partir del dataset original, se seleccionan únicamente las columnas relevantes para el análisis del mercado de vivienda de uso turístico.  
Esta selección se realiza para simplificar el dataset y alinearlo con los objetivos del proyecto.


In [12]:
cols = [
    "id",
    "host_id", "host_name", "host_listings_count", "host_total_listings_count",
    "neighbourhood_cleansed", "latitude", "longitude",
    "room_type", "accommodates",
    "price", "minimum_nights", "maximum_nights",
    "review_scores_rating", "review_scores_location", "review_scores_value",
    "license", "last_scraped" 
]

df = listings[cols].copy()
df.head()

,id,host_id,host_name,host_listings_count,host_total_listings_count,neighbourhood_cleansed,latitude,longitude,room_type,accommodates,price,minimum_nights,maximum_nights,review_scores_rating,review_scores_location,review_scores_value,license,last_scraped
0,18674,71615,Mireia,41.0,46.0,la Sagrada Família,41.405560,2.17262,Entire home/apt,8,$210.00,1,1125,4.34,4.82,4.32,ESFCTU000008058000039706000000000000000HUTB-00...,2025-09-15
1,23197,90417,Etain (Marnie),6.0,9.0,el Besòs i el Maresme,41.412432,2.21975,Entire home/apt,5,$285.00,3,32,4.82,4.66,4.68,ESFCTU000008106000547162000000000000000000HUTB...,2025-09-14
2,32711,135703,Nick,3.0,15.0,el Camp d'en Grassot i Gràcia Nova,41.405660,2.17015,Entire home/apt,6,$170.00,1,31,4.46,4.89,4.47,HUTB-001722,2025-09-15
3,34241,73163,Andres,5.0,5.0,el Barri Gòtic,41.380620,2.17517,Entire home/apt,2,$110.00,31,180,4.36,4.73,4.23,Exempt,2025-09-15
4,34981,73163,Andres,5.0,5.0,el Barri Gòtic,41.379780,2.17623,Entire home/apt,9,$333.00,5,365,4.57,4.65,4.46,ESFCTU000008119000093652000000000000000HUTB-00...,2025-09-15


In [13]:
df.isna().sum()

id                              0
host_id                         0
host_name                       3
host_listings_count             3
host_total_listings_count       3
neighbourhood_cleansed          0
latitude                        0
longitude                       0
room_type                       0
accommodates                    0
price                        1521
minimum_nights                  0
maximum_nights                  0
review_scores_rating         3944
review_scores_location       3944
review_scores_value          3944
license                      3995
last_scraped                    0
dtype: int64

### Conversión de fecha de extracción

La variable `last_scraped` se convierte a formato de fecha para facilitar su correcta interpretación temporal dentro del análisis.

In [14]:
df['last_scraped'] = pd.to_datetime(df['last_scraped'])
print("Fecha mínima:", df['last_scraped'].min())
print("Fecha máxima:", df['last_scraped'].max())

Fecha mínima: 2025-09-14 00:00:00
Fecha máxima: 2025-09-15 00:00:00


### Limpieza y transformación de la variable precio

La variable `price` se encuentra originalmente como texto y contiene símbolos de moneda.  
Se realiza su limpieza y conversión a formato numérico para permitir análisis posteriores.  
Los anuncios sin precio informado se conservan, ya que constituyen un subsegmento relevante del mercado.


In [15]:
df["price"] = (
    df["price"].astype(str)
      .str.replace("$", "", regex=False)
      .str.replace("€", "", regex=False)
      .str.replace(",", "", regex=False)
      .str.strip()
)

df["price"] = pd.to_numeric(df["price"], errors="coerce")


In [16]:
df["host_listings_count"] = df["host_listings_count"].fillna(0).astype(int)
df["host_total_listings_count"] = df["host_total_listings_count"].fillna(0).astype(int)

### Creación de variables derivadas

Se crean nuevas variables que permiten enriquecer el análisis:
- Precio por persona, como indicador de posicionamiento
- Tipo de host, para distinguir entre propietarios individuales y profesionales
Estas variables no modifican los datos originales, sino que aportan nuevas dimensiones analíticas.


In [17]:
df["price_per_person"] = df["price"] / df["accommodates"]

In [18]:
df["price_per_person"] = df["price_per_person"].round(2)

In [19]:
df["host_type"] = df["host_total_listings_count"].apply(
    lambda x: "professional" if x > 1 else "individual"
)

In [20]:
df.head()

,id,host_id,host_name,host_listings_count,host_total_listings_count,neighbourhood_cleansed,latitude,longitude,room_type,accommodates,price,minimum_nights,maximum_nights,review_scores_rating,review_scores_location,review_scores_value,license,last_scraped,price_per_person,host_type
0,18674,71615,Mireia,41,46,la Sagrada Família,41.405560,2.17262,Entire home/apt,8,210.0,1,1125,4.34,4.82,4.32,ESFCTU000008058000039706000000000000000HUTB-00...,2025-09-15,26.25,professional
1,23197,90417,Etain (Marnie),6,9,el Besòs i el Maresme,41.412432,2.21975,Entire home/apt,5,285.0,3,32,4.82,4.66,4.68,ESFCTU000008106000547162000000000000000000HUTB...,2025-09-14,57.00,professional
2,32711,135703,Nick,3,15,el Camp d'en Grassot i Gràcia Nova,41.405660,2.17015,Entire home/apt,6,170.0,1,31,4.46,4.89,4.47,HUTB-001722,2025-09-15,28.33,professional
3,34241,73163,Andres,5,5,el Barri Gòtic,41.380620,2.17517,Entire home/apt,2,110.0,31,180,4.36,4.73,4.23,Exempt,2025-09-15,55.00,professional
4,34981,73163,Andres,5,5,el Barri Gòtic,41.379780,2.17623,Entire home/apt,9,333.0,5,365,4.57,4.65,4.46,ESFCTU000008119000093652000000000000000HUTB-00...,2025-09-15,37.00,professional


In [21]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 16676 entries, 0 to 19409
Data columns (total 20 columns):
 #   Column                     Non-Null Count  Dtype         
---  ------                     --------------  -----         
 0   id                         16676 non-null  int64         
 1   host_id                    16676 non-null  int64         
 2   host_name                  16673 non-null  object        
 3   host_listings_count        16676 non-null  int32         
 4   host_total_listings_count  16676 non-null  int32         
 5   neighbourhood_cleansed     16676 non-null  object        
 6   latitude                   16676 non-null  float64       
 7   longitude                  16676 non-null  float64       
 8   room_type                  16676 non-null  object        
 9   accommodates               16676 non-null  int64         
 10  price                      15155 non-null  float64       
 11  minimum_nights             16676 non-null  int64         
 12  maximum_n

### Análisis de anuncios con y sin precio

Se identifican y analizan de forma separada los anuncios que no presentan precio informado.  
El objetivo es determinar si estos anuncios corresponden a errores de datos o si conforman un subsegmento diferenciado del mercado.


In [22]:
df_with_price = df[df["price"].notna()]
df_no_price = df[df["price"].isna()]

### Comparación entre anuncios con y sin precio

A continuación, se presentan tablas comparativas que analizan diferencias entre anuncios con y sin precio en términos de:
- Volumen de anuncios
- Distribución territorial
- Tipo de alojamiento
- Situación de licencia


In [23]:
total = df.shape[0]
sin_precio = df["price"].isna().sum()
con_precio = df["price"].notna().sum()

tabla_1 = pd.DataFrame({
    "Grupo": ["Con precio", "Sin precio"],
    "Cantidad de anuncios": [con_precio, sin_precio],
    "Porcentaje sobre el total": [
        round(con_precio / total * 100, 2),
        round(sin_precio / total * 100, 2)
    ]
})

tabla_1

,Grupo,Cantidad de anuncios,Porcentaje sobre el total
0,Con precio,15155,90.88
1,Sin precio,1521,9.12


Casi un 10% de los anuncios no tiene precio informado, por lo que no se trata de casos aislados sino de un subgrupo relevante del mercado (tener en cuenta que del dataset original ya se habían eliminado otros registros sin precio, fundamentado en la decisión de quitar aquellos con disponibilidad = 0)

In [24]:
tabla_2 = (
    df_no_price["neighbourhood_cleansed"]
    .value_counts()
    .reset_index()
)

tabla_2.columns = ["Barrio", "Cantidad de anuncios sin precio"]

tabla_2.head(10)

,Barrio,Cantidad de anuncios sin precio
0,la Dreta de l'Eixample,271
1,el Raval,176
2,el Barri Gòtic,134
3,l'Antiga Esquerra de l'Eixample,113
4,"Sant Pere, Santa Caterina i la Ribera",95
5,la Sagrada Família,76
6,Sant Antoni,74
7,la Vila de Gràcia,69
8,la Nova Esquerra de l'Eixample,65
9,la Barceloneta,61


Los anuncios sin precio se concentran en los mismos barrios centrales y con alta oferta turística que el resto del mercado, lo que indica que forman parte del mercado principal y no de zonas marginales.

In [25]:
tabla_3 = pd.DataFrame({
    "Con precio (%)": (
        df_with_price["room_type"]
        .value_counts(normalize=True) * 100
    ).round(2),

    "Sin precio (%)": (
        df_no_price["room_type"]
        .value_counts(normalize=True) * 100
    ).round(2)
}).fillna(0)

tabla_3

,Con precio (%),Sin precio (%)
room_type,,
Entire home/apt,68.66,46.88
Hotel room,0.32,1.45
Private room,30.32,51.68
Shared room,0.70,0.00


Mientras que los anuncios con precio están dominados por viviendas completas, los anuncios sin precio corresponden mayoritariamente a habitaciones privadas, lo que sugiere un uso más residencial que turístico.

In [26]:
df_with_price = df[df["price"].notna()]
df_no_price = df[df["price"].isna()]

### Análisis de la variable licencia

La variable `license` presenta una alta diversidad de formatos y valores.  
Por este motivo, se crea una clasificación simplificada que permite analizar de forma agregada la situación regulatoria de los anuncios.


In [27]:
df["license"].str.lower().value_counts().head(10)

license
exempt         3656
hutb-000000      80
hutb012407       23
aj000593         22
hb-004525        17
hutb-012457      17
hb004232         16
hb-004002        14
aj000680         14
aj000517         14
Name: count, dtype: int64

In [28]:
def classify_license(x):
    if pd.isna(x):
        return "No license"
    elif str(x).strip().lower() == "exempt":
        return "Exempt"
    else:
        return "Licensed"

df["license_type"] = df["license"].apply(classify_license)

In [29]:
tabla_license = pd.DataFrame({
    "Cantidad": df["license_type"].value_counts(),
    "Porcentaje (%)": (
        df["license_type"].value_counts(normalize=True) * 100
    ).round(2)
})

tabla_license

,Cantidad,Porcentaje (%)
license_type,,
Licensed,9025,54.12
No license,3995,23.96
Exempt,3656,21.92


In [30]:
df_with_price = df[df["price"].notna()]
df_no_price = df[df["price"].isna()]

In [31]:
tabla_license_compare = pd.DataFrame({
    "Con precio (%)": (
        df_with_price["license_type"]
        .value_counts(normalize=True) * 100
    ).round(2),

    "Sin precio (%)": (
        df_no_price["license_type"]
        .value_counts(normalize=True) * 100
    ).round(2)
}).fillna(0)

tabla_license_compare

,Con precio (%),Sin precio (%)
license_type,,
Licensed,54.71,48.19
No license,23.05,33.00
Exempt,22.24,18.80


Al excluir los anuncios con `availability_365 = 0`, la distribución del tipo de licencia entre los anuncios sin precio se aproxima más a la observada en el mercado activo. En particular, disminuye la proporción de anuncios sin licencia declarada, lo que indica que gran parte del comportamiento menos regulado estaba concentrado en registros sin disponibilidad anual.


In [32]:
df = df.rename(columns={
    "id": "id_anuncio",
    "host_id": "id_host",
    "host_name": "nombre_host",
    "host_listings_count": "cantidad_anuncios_host",
    "host_total_listings_count": "cantidad_total_anuncios_host",
    "neighbourhood_cleansed": "barrio",
    "latitude": "latitud",
    "longitude": "longitud",
    "room_type": "tipo_alojamiento",
    "accommodates": "capacidad",
    "price": "precio",
    "minimum_nights": "noches_minimas",
    "maximum_nights": "noches_maximas",
    "review_scores_rating": "puntuacion_general",
    "review_scores_location": "puntuacion_ubicacion",
    "review_scores_value": "puntuacion_valor",
    "license": "licencia",
    "price_per_person": "precio_por_persona",
    "host_type": "tipo_host",
    "license_type": "tipo_licencia",
    "last_scraped ": "ultimo_scrapeo"
})


Se renombraron las columnas al castellano.

In [33]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 16676 entries, 0 to 19409
Data columns (total 21 columns):
 #   Column                        Non-Null Count  Dtype         
---  ------                        --------------  -----         
 0   id_anuncio                    16676 non-null  int64         
 1   id_host                       16676 non-null  int64         
 2   nombre_host                   16673 non-null  object        
 3   cantidad_anuncios_host        16676 non-null  int32         
 4   cantidad_total_anuncios_host  16676 non-null  int32         
 5   barrio                        16676 non-null  object        
 6   latitud                       16676 non-null  float64       
 7   longitud                      16676 non-null  float64       
 8   tipo_alojamiento              16676 non-null  object        
 9   capacidad                     16676 non-null  int64         
 10  precio                        15155 non-null  float64       
 11  noches_minimas                166

In [34]:
df.head()

,id_anuncio,id_host,nombre_host,cantidad_anuncios_host,cantidad_total_anuncios_host,barrio,latitud,longitud,tipo_alojamiento,capacidad,...,noches_minimas,noches_maximas,puntuacion_general,puntuacion_ubicacion,puntuacion_valor,licencia,last_scraped,precio_por_persona,tipo_host,tipo_licencia
0,18674,71615,Mireia,41,46,la Sagrada Família,41.405560,2.17262,Entire home/apt,8,...,1,1125,4.34,4.82,4.32,ESFCTU000008058000039706000000000000000HUTB-00...,2025-09-15,26.25,professional,Licensed
1,23197,90417,Etain (Marnie),6,9,el Besòs i el Maresme,41.412432,2.21975,Entire home/apt,5,...,3,32,4.82,4.66,4.68,ESFCTU000008106000547162000000000000000000HUTB...,2025-09-14,57.00,professional,Licensed
2,32711,135703,Nick,3,15,el Camp d'en Grassot i Gràcia Nova,41.405660,2.17015,Entire home/apt,6,...,1,31,4.46,4.89,4.47,HUTB-001722,2025-09-15,28.33,professional,Licensed
3,34241,73163,Andres,5,5,el Barri Gòtic,41.380620,2.17517,Entire home/apt,2,...,31,180,4.36,4.73,4.23,Exempt,2025-09-15,55.00,professional,Exempt
4,34981,73163,Andres,5,5,el Barri Gòtic,41.379780,2.17623,Entire home/apt,9,...,5,365,4.57,4.65,4.46,ESFCTU000008119000093652000000000000000HUTB-00...,2025-09-15,37.00,professional,Licensed


### Conclusión del análisis del dataset listings

El análisis del dataset de anuncios (listings) de Airbnb en Barcelona permitió comprender la estructura de la oferta y detectar patrones relevantes asociados al tipo de alojamiento, la localización, el perfil del host y la situación regulatoria.

Se identificó que una proporción significativa de los anuncios no presenta precio informado. Lejos de tratarse de errores o registros aislados, estos anuncios conforman un subsegmento claramente diferenciado del mercado. En términos territoriales, se concentran en los mismos barrios centrales y de alta demanda que el resto de la oferta, lo que indica que forman parte del núcleo del mercado urbano.

El análisis del tipo de alojamiento muestra que los anuncios sin precio corresponden mayoritariamente a habitaciones privadas, mientras que los anuncios con precio están dominados por viviendas completas. Asimismo, se observa que los anuncios sin precio presentan una mayor proporción de ausencia de licencia declarada, lo que sugiere una menor alineación con el mercado turístico regulado y una posible vinculación con usos residenciales o de larga estancia.

Dado que estos anuncios aportan información relevante sobre dinámicas de informalidad, adaptación regulatoria y tensión en el mercado de vivienda, no fueron eliminados del dataset. En su lugar, se mantuvieron y se analizaron como un subgrupo específico, utilizando filtros únicamente cuando el análisis lo requirió.

Finalmente, el dataset fue limpiado, enriquecido con variables derivadas y renombrado en castellano, quedando preparado para su almacenamiento en una base de datos relacional y su posterior análisis conjunto con el dataset calendar.


### Exportación del dataset final

Una vez finalizada la limpieza, transformación y análisis exploratorio del dataset de anuncios, se exporta el dataframe final a formato CSV.  
Este archivo será utilizado para su posterior carga en una base de datos MySQL y para el análisis conjunto con el dataset calendar.


In [35]:
df.to_csv("listings_clean.csv", index=False)

### Subconjunto para análisis de precios

Para los análisis específicamente relacionados con precios (por ejemplo, distribución de precios, outliers o comparaciones de posicionamiento), se crea un subconjunto del dataset que incluye únicamente anuncios con precio válido.

Este subconjunto excluye:
- anuncios sin precio informado
- anuncios con precio nulo o igual a cero

El objetivo es evitar distorsiones en los análisis de pricing, sin eliminar estos anuncios del dataset principal, ya que siguen siendo relevantes para otros análisis de mercado y regulación.


In [36]:
listings_2 = df[df["precio"].notna() & (df["precio"] > 0)].copy()

In [37]:
listings_2.head()

,id_anuncio,id_host,nombre_host,cantidad_anuncios_host,cantidad_total_anuncios_host,barrio,latitud,longitud,tipo_alojamiento,capacidad,...,noches_minimas,noches_maximas,puntuacion_general,puntuacion_ubicacion,puntuacion_valor,licencia,last_scraped,precio_por_persona,tipo_host,tipo_licencia
0,18674,71615,Mireia,41,46,la Sagrada Família,41.405560,2.17262,Entire home/apt,8,...,1,1125,4.34,4.82,4.32,ESFCTU000008058000039706000000000000000HUTB-00...,2025-09-15,26.25,professional,Licensed
1,23197,90417,Etain (Marnie),6,9,el Besòs i el Maresme,41.412432,2.21975,Entire home/apt,5,...,3,32,4.82,4.66,4.68,ESFCTU000008106000547162000000000000000000HUTB...,2025-09-14,57.00,professional,Licensed
2,32711,135703,Nick,3,15,el Camp d'en Grassot i Gràcia Nova,41.405660,2.17015,Entire home/apt,6,...,1,31,4.46,4.89,4.47,HUTB-001722,2025-09-15,28.33,professional,Licensed
3,34241,73163,Andres,5,5,el Barri Gòtic,41.380620,2.17517,Entire home/apt,2,...,31,180,4.36,4.73,4.23,Exempt,2025-09-15,55.00,professional,Exempt
4,34981,73163,Andres,5,5,el Barri Gòtic,41.379780,2.17623,Entire home/apt,9,...,5,365,4.57,4.65,4.46,ESFCTU000008119000093652000000000000000HUTB-00...,2025-09-15,37.00,professional,Licensed


In [38]:
listings_2.info()

<class 'pandas.core.frame.DataFrame'>
Index: 15155 entries, 0 to 19409
Data columns (total 21 columns):
 #   Column                        Non-Null Count  Dtype         
---  ------                        --------------  -----         
 0   id_anuncio                    15155 non-null  int64         
 1   id_host                       15155 non-null  int64         
 2   nombre_host                   15152 non-null  object        
 3   cantidad_anuncios_host        15155 non-null  int32         
 4   cantidad_total_anuncios_host  15155 non-null  int32         
 5   barrio                        15155 non-null  object        
 6   latitud                       15155 non-null  float64       
 7   longitud                      15155 non-null  float64       
 8   tipo_alojamiento              15155 non-null  object        
 9   capacidad                     15155 non-null  int64         
 10  precio                        15155 non-null  float64       
 11  noches_minimas                151

In [39]:
listings_2.isna().sum()

id_anuncio                         0
id_host                            0
nombre_host                        3
cantidad_anuncios_host             0
cantidad_total_anuncios_host       0
barrio                             0
latitud                            0
longitud                           0
tipo_alojamiento                   0
capacidad                          0
precio                             0
noches_minimas                     0
noches_maximas                     0
puntuacion_general              3451
puntuacion_ubicacion            3451
puntuacion_valor                3451
licencia                        3493
last_scraped                       0
precio_por_persona                 0
tipo_host                          0
tipo_licencia                      0
dtype: int64

Se creó un subconjunto del dataset exclusivamente para análisis de precios, con el fin de evitar distorsiones, sin eliminar del análisis general los anuncios sin precio. Se evaluará luego la utilización de este subconjunto dentro de la información a guardar en la base de datos y del proyecto a realizar.